# Check Your Answer

**What this does:** you solve a differential equation by hand, type your answer here, and it tells
you whether your answer actually satisfies the equation.

**What this does not do:** solve anything. It has no idea what the answer is. It can only test
something you give it, which makes it useless for doing your homework and genuinely useful for
catching the dropped sign that would have cost you the problem.

Use it on any first-order equation, any method. Separable, linear, exact, integrating factors, it
does not care how you got there.

**Two ways it can tell you are wrong**, and they are different mistakes worth separating:

1. Your formula does not satisfy the equation at all. The method went wrong somewhere.
2. Your formula satisfies the equation, but your constant $C$ does not match the initial
   condition. The method was right and the last step was not.

New to notebooks? Do `00-start-here.ipynb` first.

## Run this first

Long cell, and you never need to read a line of it. Run it once (**Shift + Enter**) and give it a
moment the first time.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

_FUNCS = {
    'exp': np.exp, 'log': np.log, 'ln': np.log, 'log10': np.log10,
    'sin': np.sin, 'cos': np.cos, 'tan': np.tan,
    'sec': lambda x: 1/np.cos(x), 'csc': lambda x: 1/np.sin(x),
    'cot': lambda x: 1/np.tan(x),
    'sinh': np.sinh, 'cosh': np.cosh, 'tanh': np.tanh,
    'arctan': np.arctan, 'arcsin': np.arcsin, 'arccos': np.arccos,
    'sqrt': np.sqrt, 'abs': np.abs, 'pi': np.pi, 'e': np.e,
}


class MathTypo(Exception):
    pass


def _clean(expr, name='answer'):
    'Turn what a student typed into something Python can evaluate.'
    s = str(expr).strip()

    # Drop a leading "y =" and a trailing "= C", both of which students write by habit.
    if re.match(r"^\s*(y\s*'|dy\s*/\s*dt)\s*=", s):
        raise MathTypo(
            f"That looks like a derivative. Your {name} should be the formula for y itself, "
            "the thing you got after integrating, not y'."
        )
    s = re.sub(r'^\s*(y\s*\(\s*t\s*\)|y)\s*=(?!=)', '', s).strip()
    if '=' in s:
        left, right = s.split('=', 1)
        if right.strip().upper() in ('C', ''):
            s = left.strip()
        else:
            raise MathTypo(
                f"There is still an '=' in your {name}. Type only the side that has t and y in it, "
                "not the whole equation."
            )

    s = s.replace('^', '**')

    # 3y and 2(t+1) are the two things everyone types and Python rejects.
    m = re.search(r'(\d)\s*([a-df-zA-DF-Z(])', s)
    if m:
        raise MathTypo(
            f"Python needs an explicit * for multiplication, so write {m.group(1)}*{m.group(2)} "
            f"rather than {m.group(1)}{m.group(2)}."
        )
    if re.search(r'\)\s*[\w(]', s):
        raise MathTypo("Put a * between a closing parenthesis and whatever multiplies it.")
    if not s:
        raise MathTypo(f"Your {name} came through empty.")
    return s


def _compile(expr, name='answer'):
    src = _clean(expr, name)
    try:
        codeobj = compile(src, '<answer>', 'eval')
    except SyntaxError:
        raise MathTypo(
            f"Python could not read your {name}. Check for a missing parenthesis, and remember "
            "** for powers, * for multiplication, and exp( ) rather than e^."
        )

    def fn(**kw):
        ns = dict(_FUNCS)
        ns.update(kw)
        with np.errstate(all='ignore'):
            try:
                out = eval(codeobj, {'__builtins__': {}}, ns)
            except NameError as err:
                bad = str(err).split("'")[1] if "'" in str(err) else '?'
                raise MathTypo(
                    f"'{bad}' is not something I recognize in your {name}. Use t for the variable, "
                    "y where a y belongs, C for the constant, and exp/log/sin/cos/sqrt for functions."
                )
        return np.asarray(out, dtype=float) * np.ones_like(
            next((v for v in kw.values() if np.ndim(v) > 0), 1.0), dtype=float)
    return fn, src


def _uses(src, var):
    return re.search(r'\b' + var + r'\b', src) is not None


def _mask_poles(ys, y_range):
    'Blank out the vertical jumps so a pole does not draw a fake line across the plot.'
    span = y_range[1] - y_range[0]
    ys = np.where(np.abs(ys) > y_range[1] + 3 * span, np.nan, ys)
    ys = np.where(np.abs(ys) < y_range[0] - 3 * span, np.nan, ys)
    d = np.abs(np.diff(ys))
    jump = np.concatenate([[False], d > span])
    return np.where(jump, np.nan, ys)


def _field(ax, f, t_range, y_range, density=21):
    t = np.linspace(*t_range, density)
    y = np.linspace(*y_range, density)
    T, Y = np.meshgrid(t, y)
    with np.errstate(all='ignore'):
        M = f(t=T, y=Y)
    M = np.where(np.isfinite(M), M, np.nan)
    U = 1 / np.sqrt(1 + M**2)
    V = M / np.sqrt(1 + M**2)
    ax.quiver(T, Y, U, V, angles='xy', pivot='middle', color='0.35',
              headlength=0, headwidth=0, headaxislength=0)
    ax.set_xlim(t_range)
    ax.set_ylim(y_range)
    ax.set_xlabel('t')
    ax.set_ylabel('y')
    ax.grid(alpha=0.2)


TOL = 1e-5


def check(ode, answer, C=None, t0=None, y0=None,
          t_range=(-3, 3), y_range=(-3, 3)):
    """Test a by-hand solution against y' = ode. Does not solve anything for you."""
    try:
        f, ode_src = _compile(ode, 'equation')
        g, ans_src = _compile(answer, 'answer')
    except MathTypo as err:
        print('Could not read that.\n\n' + str(err))
        return

    implicit = _uses(ans_src, 'y')
    has_C = _uses(ans_src, 'C')

    print(f"equation:  y' = {ode_src}")
    print(f"answer:    {'F(t,y) = C  with F = ' if implicit else 'y = '}{ans_src}")
    if implicit:
        print('           (read as an implicit answer, because there is a y in it)')
    print()

    try:
        if implicit:
            ok = _check_implicit(f, g, has_C, t_range, y_range, ode_src)
        else:
            ok = _check_explicit(f, g, has_C, C, t_range, y_range, ode_src)
        if ok and t0 is not None and y0 is not None:
            _check_ic(g, implicit, has_C, C, t0, y0)
    except MathTypo as err:
        print('Could not evaluate that.\n\n' + str(err))


def _residual_explicit(f, g, t, C=None):
    h = 1e-6
    kw = {} if C is None else {'C': C}
    with np.errstate(all='ignore'):
        gp = (g(t=t + h, **kw) - g(t=t - h, **kw)) / (2 * h)
        gv = g(t=t, **kw)
        rhs = f(t=t, y=gv)
    return gp - rhs, rhs, gv


def _verdict(worst, where, ode_src, frac_ok=0.0, ok_span=None):
    if worst < TOL:
        print(f"MATCH. This satisfies y' = {ode_src}.")
        print(f"       Largest mismatch anywhere sampled: {worst:.2e}, which is round-off.")
        return True
    print(f"NO MATCH. This does not satisfy y' = {ode_src}.")
    print(f"          Largest mismatch: {worst:.2e}, near t = {where:.2f}.")
    if frac_ok > 0.25:
        print()
        print(f"          But it does match on about {100*frac_ok:.0f}% of the window"
              + (f", roughly {ok_span[0]:.2f} < t < {ok_span[1]:.2f}." if ok_span else "."))
        print("          That pattern usually means the answer is right but only on part of the")
        print("          domain. Worth working out where it stops being valid, and why.")
    else:
        print("          In the plot, look for your curve cutting across the segments instead of")
        print("          riding along them. That is the same failure the number is reporting.")
    return False


def _check_explicit(f, g, has_C, C, t_range, y_range, ode_src):
    t = np.linspace(t_range[0], t_range[1], 801)
    Cs = [C] if C is not None else ([-2.0, -0.5, 0.5, 2.0] if has_C else [None])
    if not has_C:
        Cs = [None]
    elif C is not None:
        Cs = [C]

    fig, ax = plt.subplots(figsize=(8, 5))
    _field(ax, f, t_range, y_range)

    worst, where, drawn, sampled = 0.0, np.nan, 0, 0
    n_ok, n_tot, ok_span = 0, 0, None
    for i, c in enumerate(Cs):
        resid, rhs, gv = _residual_explicit(f, g, t, c)
        good = np.isfinite(resid) & np.isfinite(gv) & (np.abs(gv) < 1e6)
        if good.any():
            rel = np.abs(resid[good]) / (1 + np.abs(rhs[good]))
            sampled += int(good.sum())
            n_ok += int((rel < TOL).sum())
            n_tot += int(rel.size)
            if i == 0 and (rel < TOL).any():
                tg = t[good][rel < TOL]
                ok_span = (float(tg.min()), float(tg.max()))
            k = int(np.argmax(rel))
            if rel[k] > worst:
                worst, where = float(rel[k]), float(t[good][k])
        lab = 'your answer' if c is None else f'C = {c:g}'
        ys = _mask_poles(np.where(good, gv, np.nan), y_range)
        if np.isfinite(ys).any():
            ax.plot(t, ys, linewidth=2.5, color=f'C{i}', label=lab,
                    linestyle=['-', '--', '-.', ':'][i % 4])
            drawn += 1

    ax.set_title(f"y' = {ode_src}")
    if drawn:
        ax.legend(loc='upper left', framealpha=0.9)
    plt.show()
    plt.close(fig)

    if sampled < 20:
        print('Your answer is undefined almost everywhere in this window, so there is nothing')
        print('to compare. Try widening t_range/y_range, or check for a typo.')
        return False
    return _verdict(worst, where, ode_src,
                    frac_ok=(n_ok / n_tot if n_tot else 0.0), ok_span=ok_span)


def _check_implicit(f, g, has_C, t_range, y_range, ode_src):
    h = 1e-6
    n = 60
    t = np.linspace(t_range[0], t_range[1], n)
    y = np.linspace(y_range[0], y_range[1], n)
    T, Y = np.meshgrid(t, y)
    kw = {'C': 0.0} if has_C else {}

    with np.errstate(all='ignore'):
        Ft = (g(t=T + h, y=Y, **kw) - g(t=T - h, y=Y, **kw)) / (2 * h)
        Fy = (g(t=T, y=Y + h, **kw) - g(t=T, y=Y - h, **kw)) / (2 * h)
        slope = -Ft / Fy
        rhs = f(t=T, y=Y)
        F = g(t=T, y=Y, **kw)

    good = (np.isfinite(slope) & np.isfinite(rhs) & (np.abs(Fy) > 1e-4)
            & (np.abs(slope) < 1e4) & (np.abs(rhs) < 1e4))

    fig, ax = plt.subplots(figsize=(8, 5))
    _field(ax, f, t_range, y_range)
    if np.isfinite(F).any():
        try:
            cs = ax.contour(T, Y, F, levels=9, colors='C0', linewidths=2.0)
            ax.clabel(cs, inline=True, fontsize=8, fmt='C=%.1f')
        except Exception:
            pass
    ax.set_title(f"y' = {ode_src}   with your level curves F(t,y) = C")
    plt.show()
    plt.close(fig)

    if good.sum() < 50:
        print('Could not compare on enough of this window. Try a different t_range/y_range.')
        return False

    rel = np.abs(slope[good] - rhs[good]) / (1 + np.abs(rhs[good]))
    worst = float(np.max(rel))
    k = int(np.argmax(rel))
    where = float(T[good][k])

    if worst < 1e-4:
        print(f"MATCH. Along your level curves F(t,y) = C, the slope is exactly {ode_src}.")
        print(f"       Largest mismatch anywhere sampled: {worst:.2e}, which is round-off.")
        print('       Each labelled curve above is one value of C. The initial condition picks one.')
        return True
    print(f"NO MATCH. Along your level curves, the slope is not {ode_src}.")
    print(f"          Largest mismatch: {worst:.2e}, near t = {where:.2f}.")
    print('          Compare a labelled curve against the segments it crosses.')
    return False


def _check_ic(g, implicit, has_C, C, t0, y0):
    print()
    if implicit:
        if C is None:
            print(f'Initial condition: F({t0:g}, {y0:g}) tells you which C you need.')
            print(f'                   Your C should be {float(g(t=np.array([float(t0)]), y=np.array([float(y0)]), C=0.0)[0]):.6g}.')
        return
    if not has_C:
        val = float(g(t=np.array([float(t0)]))[0])
    elif C is None:
        print('Initial condition: not checked, because you did not say what C you got.')
        print('                   Pass C=... to check it.')
        return
    else:
        val = float(g(t=np.array([float(t0)]), C=float(C))[0])
    gap = abs(val - float(y0))
    print(f'Initial condition y({t0:g}) = {y0:g}:')
    if gap < 1e-8:
        print(f'  MATCH. Your answer gives y({t0:g}) = {val:.6g}.')
    else:
        print(f'  NO MATCH. Your answer gives y({t0:g}) = {val:.6g}, but you wanted {float(y0):.6g}.')
        print('  If the family matched above, the method is right and only the constant is off.')

print('Ready. Scroll down.')

## How to type your answer

Typing math for a computer is fussier than writing it on paper. Six rules cover everything:

| On paper | Type this |
| --- | --- |
| $y^2$ | `y^2` or `y**2` |
| $3y$ | `3*y`, the `*` is required |
| $e^t$ | `exp(t)` |
| $e^{-2t}$ | `exp(-2*t)` |
| $\ln t$ | `ln(t)` or `log(t)` |
| $\sqrt{y}$ | `sqrt(y)` |
| $\frac{t+1}{t-1}$ | `(t+1)/(t-1)` |

Use `t` for the independent variable, `y` where a $y$ belongs, and `C` for the arbitrary constant.
`sin`, `cos`, `tan`, `arctan`, and `pi` all work too.

If you mistype something it will tell you what to fix in plain English. It will not hand you a
wall of red.

## What a correct answer looks like

Take $y'=y$. Separating and integrating gives $y=Ce^{t}$. Run this.

In [ ]:
check(ode="y", answer="C*exp(t)")

Two things came back. The **number** is the real verdict: it substituted your formula into the
equation and measured how badly the two sides disagree. Something like `1e-10` means they agree
exactly and the leftovers are round-off.

The **picture** is the same verdict in a form you can see. Your curves ride along the segments of
the slope field instead of cutting across them, which is what it means to solve the equation.

Each dashed and dotted curve is a different value of $C$. That is the family. An initial condition
picks one out of it.

## What a wrong answer looks like

Worth seeing once, so you trust the thing when it disagrees with you.

Same equation, but suppose you slipped and wrote $y=Ce^{2t}$.

In [ ]:
check(ode="y", answer="C*exp(2*t)")

The mismatch is around `1`, not `1e-10`, and in the picture the curves climb straight through the
segments at the wrong angle. Once you have seen a wrong one, the difference is obvious at a glance.

## Your turn

This is the cell to keep coming back to. Change the two strings and run it. Nothing else.

Change `ode` to the right side of your equation, written as $y'=\,$ something. Change `answer` to
what you got.

In [ ]:
check(
    ode="t - y",                  # your equation, as y' = this
    answer="t - 1 + C*exp(-t)",   # what you got by hand
)

If your solution lives outside the default window, widen it:

```python
check(ode="...", answer="...", t_range=(-1, 6), y_range=(-10, 10))
```

Do that whenever the picture comes back empty, or when the verdict says your answer is undefined
almost everywhere.

## Answers you could not solve for $y$

Separable equations often strand you at something like

$$t^{2}+y^{2}=C,$$

with no way to isolate $y$. That is still a complete, correct answer, and this checks it too.

Type the left side, the part with $t$ and $y$ in it. Because there is a $y$ in what you typed, it
knows to read it as an implicit answer.

In [ ]:
check(ode="-t/y", answer="t^2 + y^2")

Instead of drawing one curve per $C$, it now draws the labelled level curves of your $F(t,y)$ and
checks that the slope along every one of them matches the equation.

Look at what the picture is telling you about the answer. The curves are circles, so no single
formula $y=g(t)$ could ever describe one. An initial condition puts you on one circle and on the
**top or bottom half of it**, and your solution ends where the circle turns vertical, because the
slope is undefined there. That is real information about the solution, and it is completely
invisible in the algebra you did to get $t^2+y^2=C$.

## Checking your constant

Everything above tests the *family*. To test the specific solution, tell it the initial condition
and the $C$ you solved for.

For $y'=y^{2}$ with $y(0)=1$: separating gives $y=-1/(t+C)$, and $y(0)=1$ forces $C=-1$.

In [ ]:
check(ode="y^2", answer="-1/(t+C)", C=-1, t0=0, y0=1)

Now change `C=-1` to `C=2` and run it again.

The family still matches, because the family never depended on $C$. Only the initial condition
line fails. That is the whole point of separating these two checks: it tells you whether you broke
the method or just the arithmetic at the end.

While you are here, look at the curve for $C=-1$ in the picture. It runs off to infinity at $t=1$.
Your solution does not merely get large there, it **stops existing**, and the formula gave you no
warning. Hold that thought for Section 2.3.

## It says NO MATCH. Now what?

It will not tell you the answer. Here is how to use what it does tell you.

**Check that you typed the equation right.** Easily the most common cause. Read the two lines it
echoes back at the top against your paper. A missing set of parentheses around a numerator is the
usual culprit.

**Look at where the curve leaves the segments.** If it rides along them for a while and then peels
off, the error is somewhere that only bites in part of the plane. If it is at the wrong angle
everywhere, the error is in the main body of the work.

**If it says your answer matches on part of the window**, that is a real message and not a
near-miss. Your answer is probably correct on a restricted domain. Try $y'=2\sqrt{y}$ with
$y=t^{2}$: it matches perfectly for $t>0$ and fails for $t<0$, because $\sqrt{y}$ is never
negative but $2t$ is. Finding the boundary is worth more than getting a clean MATCH.

**Differentiate your own answer by hand and substitute it in.** The checker is doing exactly this
numerically. Doing it yourself on paper for two lines usually finds the slip faster than staring
at the plot.

**Then bring it to class or office hours.** Knowing your answer is wrong, and roughly where, is
most of a good question.